<a href="https://colab.research.google.com/github/datawrangler7798/Vector-Databases-Hands-On/blob/main/Vector_DB_%26_Embedding(VECTOR_SEARCH_FAISS_(Local)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install faiss-cpu sentence-transformers pinecone


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 12.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import faiss
import time
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

In [ ]:
#PART 1 — FAISS (LOCAL VECTOR SEARCH)
print("\n========== FAISS DEMO ==========\n")

# Small dataset
texts = [
    "Document about AI simulations.",
    "Deep learning and neural networks.",
    "Vector databases and similarity search.",
    "Graph-based indexing methods.",
    "Clustering algorithms in machine learning."
]


========== FAISS DEMO ==========



In [ ]:
## Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Convert text → vectors
embeddings = model.encode(texts)
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

In [ ]:
print(dimension)

384


In [ ]:
print(embeddings)

[[-0.07829455 -0.05609139 -0.00738311 ...  0.11145332 -0.04897677
   0.00490513]
 [-0.08722686 -0.01855165  0.05427016 ...  0.03430808 -0.02533455
  -0.01078802]
 [-0.01181242 -0.04063171 -0.04198801 ... -0.05746524  0.02757708
   0.00734559]
 [ 0.02726984  0.01366909  0.0085608  ... -0.07297622 -0.02407003
   0.00442331]
 [ 0.02063868 -0.01270108  0.01477154 ... -0.00398619 -0.02105954
  -0.00546843]]


In [ ]:
#FLAT INDEX (Exact Search)
print("---- FAISS: Flat Index ----")

flat_index = faiss.IndexFlatL2(dimension)
flat_index.add(embeddings)

query = "AI systems"
query_vector = model.encode([query]).astype("float32")

k = 4
distances, indices = flat_index.search(query_vector, k)

for i in indices[0]:
    print(texts[i])


---- FAISS: Flat Index ----
Document about AI simulations.
Deep learning and neural networks.
Clustering algorithms in machine learning.
Vector databases and similarity search.


In [ ]:
# HNSW INDEX (Graph-based)
# --------------------------------------------
print("\n---- FAISS: HNSW Index ----")

hnsw_index = faiss.IndexHNSWFlat(dimension, 32)
hnsw_index.add(embeddings)

distances, indices = hnsw_index.search(query_vector, k)

for i in indices[0]:
    print(texts[i])


---- FAISS: HNSW Index ----
Document about AI simulations.
Deep learning and neural networks.
Clustering algorithms in machine learning.
Vector databases and similarity search.


In [ ]:
#IVF INDEX (Cluster-based)
# --------------------------------------------
print("\n---- FAISS: IVF Index ----")

nlist = 3  # number of clusters

quantizer = faiss.IndexFlatL2(dimension)
ivf_index = faiss.IndexIVFFlat(quantizer, dimension, nlist)

ivf_index.train(embeddings)
ivf_index.add(embeddings)

ivf_index.nprobe = 3  # number of clusters to search

distances, indices = ivf_index.search(query_vector, k)

for i in indices[0]:
    print(texts[i])




---- FAISS: IVF Index ----
Document about AI simulations.
Deep learning and neural networks.
Clustering algorithms in machine learning.
Vector databases and similarity search.


#PART 2 — PINECONE (CLOUD VECTOR DATABASE)

In [ ]:
print("\n========== PINECONE DEMO ==========\n")

# 🔐 Replace with your API key
api_key = "pcsk_2NXbAZ_Edj6RkLpfDhzymt2YcYp9HgwCd65nQwti5GfGvgjSpHwLnV6fZyvthwLb5xTzwK"

pc = Pinecone(api_key=api_key)

index_name = "demo-index"

# Create index if it does not exist
if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Connect to index
index = pc.Index(index_name)

# Sample data
pinecone_texts = [
    "AI is transforming healthcare",
    "Robotics in industrial automation",
    "Machine learning in finance",
    "AI for simulation and modeling",
    "Cloud computing basics"
]

# Insert data
for i, text in enumerate(pinecone_texts):

    # Convert text into embedding
    embedding = model.encode(text).tolist()

    # Create metadata
    metadata = {
        "category": "ai" if "AI" in text else "other",
        "created_at": int(time.time()),
        "keywords": text.lower().split()
    }

    # Upsert into Pinecone
    index.upsert(
        vectors=[
            (f"doc-{i}", embedding, metadata)
        ]
    )

print("Data successfully inserted into Pinecone!")


========== PINECONE DEMO ==========

Data successfully inserted into Pinecone!


1️⃣ Basic Semantic Search

In [ ]:
print("---- Pinecone: Semantic Search ----")

query = "AI in simulations"
query_embedding = model.encode(query).tolist()

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True
)

for match in results["matches"]:
    print(match["id"], match["score"])
    print(match["metadata"])
    print("-----")


---- Pinecone: Semantic Search ----
doc-3 0.875126362
{'category': 'ai', 'created_at': 1789371080, 'keywords': ['ai', 'for', 'simulation', 'and', 'modeling']}
-----
doc-0 0.394708693
{'category': 'ai', 'created_at': 1789371078, 'keywords': ['ai', 'is', 'transforming', 'healthcare']}
-----
doc-1 0.324862391
{'category': 'other', 'created_at': 1789371079, 'keywords': ['robotics', 'in', 'industrial', 'automation']}
-----


2️⃣ Filter by Category

In [ ]:
print("\n---- Pinecone: Filter by Category ----")

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "category": {"$eq": "ai"}
    }
)

for match in results["matches"]:
    print(match["id"], match["metadata"])



---- Pinecone: Filter by Category ----
doc-3 {'category': 'ai', 'created_at': 1789371080, 'keywords': ['ai', 'for', 'simulation', 'and', 'modeling']}
doc-0 {'category': 'ai', 'created_at': 1789371078, 'keywords': ['ai', 'is', 'transforming', 'healthcare']}


3️⃣ Filter by Keyword

In [ ]:
print("\n---- Pinecone: Filter by Keyword ----")

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "keywords": {"$in": ["finance"]}
    }
)

for match in results["matches"]:
    print(match["id"], match["metadata"])



---- Pinecone: Filter by Keyword ----
doc-2 {'category': 'other', 'created_at': 1789371080, 'keywords': ['machine', 'learning', 'in', 'finance']}


4️⃣ Combined Filter (AND logic)

In [ ]:
print("\n---- Pinecone: Combined Filter ----")

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "$and": [
            {"category": {"$eq": "ai"}},
            {"keywords": {"$in": ["simulation"]}}
        ]
    }
)

for match in results["matches"]:
    print(match["id"], match["metadata"])


print("\n========== DONE ==========")


---- Pinecone: Combined Filter ----
doc-3 {'category': 'ai', 'created_at': 1789371080, 'keywords': ['ai', 'for', 'simulation', 'and', 'modeling']}

========== DONE ==========
